# 03 · SQL - Schema, Load & Validation (MySQL)

Three parts, all using SQL you've worked with before:

1. **Schema** - build the normalized 4-table database from `schema.sql`
2. **Load** - split the clean flat file into the tables with pandas `to_sql`
3. **Validate** - plain `SELECT … JOIN … GROUP BY` queries that check the data
   and produce the headline numbers for the slides

**Tables:** `countries` (market + GDP), `nutriscore_grades` (A–E lookup),
`products` (one row per barcode), `product_markets` (bridge: which markets sell
each product).

**Before running:** MySQL Server up, `data/products_clean.csv` present, a `.env`
file in the project root, and `pip install sqlalchemy pymysql python-dotenv`.

## 1. Setup to read connection details from `.env`

In [1]:
import os
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine, text, URL

load_dotenv(find_dotenv())   # finds .env in the project root

HOST     = os.getenv("DB_HOST", "localhost")
PORT     = int(os.getenv("DB_PORT", 3306))
USER     = os.getenv("DB_USER", "root")
PASSWORD = os.getenv("DB_PASSWORD")          # may be "" if root has no password
DB       = os.getenv("DB_NAME", "open_food_facts")

if PASSWORD is None:
    raise ValueError("DB_PASSWORD not set in .env (leave it blank only if root truly has no password)")
print(f"Config loaded — will connect as '{USER}' to database '{DB}'")

Config loaded — will connect as 'root' to database 'open_food_facts'


## 2. Connect: create the database and test the engine

`URL.create` builds the connection string safely (handles blank or special-character
passwords). The `SELECT VERSION()` at the end proves the connection actually works.

In [3]:

server = create_engine(URL.create("mysql+pymysql", username=USER, password=PASSWORD or None,
                                  host=HOST, port=PORT))
with server.connect() as conn:
    conn.execute(text(f"CREATE DATABASE IF NOT EXISTS {DB} CHARACTER SET utf8mb4"))
    conn.commit()

engine = create_engine(URL.create("mysql+pymysql", username=USER, password=PASSWORD or None,
                                  host=HOST, port=PORT, database=DB))

with engine.connect() as conn:
    v = conn.execute(text("SELECT VERSION()")).scalar()
print(f"Connected to MySQL {v} — database '{DB}' ready.")

Connected to MySQL 9.5.0 — database 'open_food_facts' ready.


## 3. Locate `schema.sql` and define a helper to run it

In [4]:
DATA_DIR = Path("../data") if Path("../data").exists() else Path("data")
CLEAN_PATH = DATA_DIR / "products_clean.csv"

SCHEMA_PATH = next((p for p in [Path("schema.sql"), Path("../schema.sql"),
                                Path("notebooks/schema.sql")] if p.exists()), None)
if SCHEMA_PATH is None:
    raise FileNotFoundError("schema.sql not found — put it next to this notebook")

def run_sql_script(engine, path):
    """Split a .sql file on ; and run each statement (the driver runs one at a time)."""
    raw = Path(path).read_text()
    lines = [ln for ln in raw.splitlines() if not ln.strip().startswith("--")]
    statements = [s.strip() for s in "\n".join(lines).split(";") if s.strip()]
    with engine.begin() as conn:
        for stmt in statements:
            conn.execute(text(stmt))
    return len(statements)

print("Clean data:", CLEAN_PATH.resolve())
print("Schema:    ", SCHEMA_PATH.resolve())

Clean data: /Users/kanakyadav/Documents/GitHub/capstone/data/products_clean.csv
Schema:     /Users/kanakyadav/Documents/GitHub/capstone/notebooks/schema.sql


## 4. Build the schema and load the data

This single cell is **safe to re-run**: `schema.sql` drops and recreates all four
tables (so no leftover rows), fills the reference tables, then we load `products`
and `product_markets`. Splitting the flat file:
- `products` = one row per unique barcode
- `product_markets` = the (code, country) pairs

In [5]:
# 1. Rebuild the schema fresh (drops + recreates all tables, loads reference data)
n = run_sql_script(engine, SCHEMA_PATH)
print(f"Schema rebuilt ({n} statements).")

# 2. Read the clean data
clean = pd.read_csv(CLEAN_PATH, dtype={"code": "string"})
clean["last_modified"] = pd.to_datetime(clean["last_modified"], errors="coerce", utc=True).dt.tz_localize(None)

product_cols = ["code", "product_name", "brands", "nutriscore_grade", "nutriscore_score",
                "nova_group", "eco_grade", "additives_n", "energy_kcal_100g", "sugars_100g",
                "fat_100g", "saturated_fat_100g", "salt_100g", "sodium_100g", "proteins_100g",
                "fiber_100g", "completeness", "last_modified"]

# 3. Split into the two tables (dedup products by barcode)
products = clean[product_cols].drop_duplicates(subset=["code"], keep="first").reset_index(drop=True)
markets  = clean[["code", "country"]].drop_duplicates().reset_index(drop=True)

# 4. Trim very long text fields (OFF brands can be huge free text)
products["brands"]       = products["brands"].str.slice(0, 255)
products["product_name"] = products["product_name"].str.slice(0, 255)

assert products["code"].duplicated().sum() == 0, "duplicate codes remain"
print(f"Unique products: {len(products):,}   |   (product, market) rows: {len(markets):,}")

# 5. NaN -> None so they land as SQL NULL, then load
products = products.astype(object).where(pd.notnull(products), None)
products.to_sql("products", engine, if_exists="append", index=False, chunksize=1000)
markets.to_sql("product_markets", engine, if_exists="append", index=False, chunksize=2000)
print("Loaded products and product_markets.")

Schema rebuilt (10 statements).
Unique products: 69,868   |   (product, market) rows: 70,000
Loaded products and product_markets.


## 5. Sanity check: row counts per table

In [6]:

for t in ["countries", "nutriscore_grades", "products", "product_markets"]:
    n = pd.read_sql(f"SELECT COUNT(*) AS n FROM {t}", engine)["n"][0]
    print(f"{t:18s}: {n:,} rows")

countries         : 10 rows
nutriscore_grades : 5 rows
products          : 69,868 rows
product_markets   : 70,000 rows


## 6. Validation & analysis queries

### 6.1 Products per country  *(Market Overview)*

In [7]:

sql = '''
SELECT c.country, c.gdp_rank_europe, COUNT(*) AS products
FROM products p
JOIN product_markets pm ON p.code = pm.code
JOIN countries c        ON pm.country = c.country
GROUP BY c.country, c.gdp_rank_europe
ORDER BY products DESC;
'''
pd.read_sql(sql, engine)

,country,gdp_rank_europe,products
0,Belgium,10,7000
1,France,3,7000
2,Germany,1,7000
3,Italy,5,7000
4,Netherlands,7,7000
5,Poland,9,7000
6,Spain,6,7000
7,Sweden,12,7000
8,Switzerland,8,7000
9,United Kingdom,2,7000


### 6.2 Nutri-Score grade mix per country (%)

In [9]:
sql = '''
SELECT c.country,
   ROUND(100.0*SUM(CASE WHEN p.nutriscore_grade='A' THEN 1 ELSE 0 END)/COUNT(*),1) AS pct_a,
   ROUND(100.0*SUM(CASE WHEN p.nutriscore_grade='B' THEN 1 ELSE 0 END)/COUNT(*),1) AS pct_b,
   ROUND(100.0*SUM(CASE WHEN p.nutriscore_grade='C' THEN 1 ELSE 0 END)/COUNT(*),1) AS pct_c,
   ROUND(100.0*SUM(CASE WHEN p.nutriscore_grade='D' THEN 1 ELSE 0 END)/COUNT(*),1) AS pct_d,
   ROUND(100.0*SUM(CASE WHEN p.nutriscore_grade='E' THEN 1 ELSE 0 END)/COUNT(*),1) AS pct_e
FROM products p
JOIN product_markets pm ON p.code = pm.code
JOIN countries c        ON pm.country = c.country
GROUP BY c.country
ORDER BY c.country;
'''
pd.read_sql(sql, engine)

,country,pct_a,pct_b,pct_c,pct_d,pct_e
0,Belgium,6.8,4.7,9.9,11.3,10.2
1,France,5.5,4.2,8.2,10.5,11.9
2,Germany,4.6,2.6,7.3,7.6,7.2
3,Italy,4.8,4.0,6.6,9.9,8.6
4,Netherlands,5.3,3.7,6.1,5.7,6.1
5,Poland,6.3,4.5,9.6,8.2,7.6
6,Spain,4.6,4.3,6.0,7.7,7.4
7,Sweden,3.9,3.2,6.7,6.9,6.6
8,Switzerland,5.3,4.3,10.7,11.0,11.1
9,United Kingdom,5.0,4.0,7.2,6.5,6.6


### 6.3 Ultra-processed (NOVA 4) share per country

In [10]:
sql = '''
SELECT c.country,
   COUNT(*) AS products,
   ROUND(100.0*SUM(CASE WHEN p.nova_group=4 THEN 1 ELSE 0 END)/COUNT(*),1) AS pct_ultra_processed
FROM products p
JOIN product_markets pm ON p.code = pm.code
JOIN countries c        ON pm.country = c.country
GROUP BY c.country
ORDER BY pct_ultra_processed DESC;
'''
pd.read_sql(sql, engine)

,country,products,pct_ultra_processed
0,Netherlands,7000,24.0
1,United Kingdom,7000,21.2
2,Poland,7000,20.6
3,Switzerland,7000,19.3
4,Belgium,7000,19.2
5,Sweden,7000,16.9
6,France,7000,16.6
7,Germany,7000,15.2
8,Spain,7000,8.4
9,Italy,7000,6.4


### 6.4 Average nutrients per 100g per country  *(Nutrition Deep-Dive)*

In [11]:
sql = '''
SELECT c.country,
   ROUND(AVG(p.sugars_100g),1)        AS avg_sugar,
   ROUND(AVG(p.salt_100g),2)          AS avg_salt,
   ROUND(AVG(p.saturated_fat_100g),1) AS avg_sat_fat,
   ROUND(AVG(p.energy_kcal_100g),0)   AS avg_energy_kcal
FROM products p
JOIN product_markets pm ON p.code = pm.code
JOIN countries c        ON pm.country = c.country
GROUP BY c.country
ORDER BY avg_sugar DESC;
'''
pd.read_sql(sql, engine)

,country,avg_sugar,avg_salt,avg_sat_fat,avg_energy_kcal
0,France,13.8,1.24,5.4,275.0
1,Switzerland,12.0,1.43,5.0,262.0
2,United Kingdom,12.0,1.27,4.5,261.0
3,Italy,11.8,1.29,4.8,293.0
4,Germany,11.7,1.28,4.9,257.0
5,Belgium,11.5,1.19,5.0,266.0
6,Poland,11.1,1.31,4.6,253.0
7,Netherlands,10.3,1.32,4.9,254.0
8,Sweden,9.7,1.60,5.1,253.0
9,Spain,9.0,1.36,5.3,281.0


### 6.5 Data-quality check - missing Nutri-Score per country

In [12]:

sql = '''
SELECT c.country,
   COUNT(*) AS products,
   SUM(CASE WHEN p.nutriscore_grade IS NULL THEN 1 ELSE 0 END) AS missing_grade,
   ROUND(100.0*SUM(CASE WHEN p.nutriscore_grade IS NULL THEN 1 ELSE 0 END)/COUNT(*),1) AS pct_missing
FROM products p
JOIN product_markets pm ON p.code = pm.code
JOIN countries c        ON pm.country = c.country
GROUP BY c.country
ORDER BY pct_missing DESC;
'''
pd.read_sql(sql, engine)

,country,products,missing_grade,pct_missing
0,Netherlands,7000,5118.0,73.1
1,Sweden,7000,5091.0,72.7
2,Germany,7000,4951.0,70.7
3,United Kingdom,7000,4950.0,70.7
4,Spain,7000,4901.0,70.0
5,Italy,7000,4626.0,66.1
6,Poland,7000,4472.0,63.9
7,France,7000,4180.0,59.7
8,Switzerland,7000,4038.0,57.7
9,Belgium,7000,3998.0,57.1


### 6.6 Does wealth track healthiness? (4-table join)

Join in the `nutriscore_grades` lookup to turn A–E into a numeric rank
(1 = healthiest), then compare the average against GDP per capita.

In [14]:

sql = '''
SELECT c.country, c.gdp_per_capita_usd,
   COUNT(*)                    AS graded_products,
   ROUND(AVG(g.grade_rank),2)  AS avg_grade_rank   -- lower = healthier
FROM products p
JOIN product_markets pm  ON p.code = pm.code
JOIN countries c         ON pm.country = c.country
JOIN nutriscore_grades g ON p.nutriscore_grade = g.grade
GROUP BY c.country, c.gdp_per_capita_usd
ORDER BY c.gdp_per_capita_usd DESC;
'''
pd.read_sql(sql, engine)

,country,gdp_per_capita_usd,graded_products,avg_grade_rank
0,Switzerland,115620,2962,3.43
1,Netherlands,73833,1882,3.13
2,Sweden,62662,1909,3.33
3,Belgium,61002,3002,3.32
4,Germany,60439,2049,3.35
5,United Kingdom,57608,2050,3.20
6,France,48930,2820,3.48
7,Italy,43270,2374,3.40
8,Spain,38290,2099,3.30
9,Poland,28374,2528,3.17
